In [1]:
import pandas as pd
import openpyxl
import ifcopenshell
import ifcopenshell.util.element
from collections import Counter
from sklearn.model_selection import train_test_split


In [2]:
# df_matrix = pd.read_excel(r'C:\Users\lucas.galicioli\ifc-classifier\data\interim\classification-matrix.xlsx')
# display(df_matrix)

In [3]:
# import ifcopenshell
# import ifcopenshell.util.element
# import pandas as pd
# import os

# # --- Caminho do arquivo IFC ---
# ifc_file_path = r"C:\Users\lucas.galicioli\Downloads\RÔGGA EMPREENDIMENTOS-BRUSQUE HOME CLUB-2025-10-16-13-35-31-469\PHN21043-PCI-PR-0001-BIM-EMB-GER SPK-R03.ifc"

# # --- Funções Auxiliares (com a função de quantidade modificada) ---

# def get_building_storey(element):
#     """ Encontra o IfcBuildingStorey no qual o elemento está contido. """
#     try:
#         spatial_container = ifcopenshell.util.element.get_container(element)
#         if spatial_container and spatial_container.is_a('IfcBuildingStorey'):
#             return spatial_container.Name
#     except Exception:
#         pass
#     return None

# def get_material_name(element):
#     """ Extrai o nome do material associado ao elemento. """
#     material = ifcopenshell.util.element.get_material(element)
#     if not material:
#         return None
#     if hasattr(material, 'Name'):
#         return material.Name
#     elif hasattr(material, 'MaterialLayers'):
#         layer_names = [
#             layer.Material.Name 
#             for layer in material.MaterialLayers 
#             if hasattr(layer, 'Material') and hasattr(layer.Material, 'Name')
#         ]
#         return ', '.join(layer_names) if layer_names else None
#     return None
    
# # --- SOLUÇÃO 2: Função de quantidade compatível com versões antigas ---
# def get_quantity_value_legacy(element, quantity_name):
#     """
#     Busca por uma quantidade específica (ex: 'Width') e retorna seu valor.
#     Esta versão é compatível com versões mais antigas do ifcopenshell sem 'get_qsets'.
#     """
#     # Itera através das relações de definição do elemento
#     for definition in getattr(element, 'IsDefinedBy', []):
#         if definition.is_a('IfcRelDefinesByProperties'):
#             prop_set = definition.RelatingPropertyDefinition
#             # Verifica se é um conjunto de quantidades (IfcElementQuantity)
#             if prop_set.is_a('IfcElementQuantity'):
#                 # Itera através das quantidades dentro do conjunto
#                 for quantity in prop_set.Quantities:
#                     if quantity.Name == quantity_name:
#                         # Extrai o valor do atributo correto (ex: LengthValue, AreaValue)
#                         value_attribute = next((attr for attr in dir(quantity) if attr.endswith('Value')), None)
#                         if value_attribute:
#                             return getattr(quantity, value_attribute)
#     return None

# # --- Processamento Principal ---

# try:
#     ifc_file = ifcopenshell.open(ifc_file_path)
#     file_name = os.path.basename(ifc_file_path)

#     element_data = []
#     products = ifc_file.by_type('IfcProduct')

#     print(f"Processando {len(products)} elementos do arquivo: {file_name}...")

#     for product in products:
#         if product.is_a('IfcOpeningElement') or product.is_a('IfcVirtualElement'):
#             continue

#         psets = ifcopenshell.util.element.get_psets(product)
#         rogga_pset = psets.get('PSET_RÔGGA', {})

#         element_info = {
#             'GlobalId': product.GlobalId,
#             'FileName': file_name,
#             'Class': product.is_a(),
#             'PredefinedType': getattr(product, 'PredefinedType', None),
#             'Name': getattr(product, 'Name', None),
#             'BuildingStorey': get_building_storey(product),
#             'Material': get_material_name(product),
#             'PSET_RÔGGA.RÔGGA_SEÇÃO': rogga_pset.get('RÔGGA_SEÇÃO', None),
#             'PSET_RÔGGA.RÔGGA_DESCRIÇÃO': rogga_pset.get('RÔGGA_DESCRIÇÃO', None),
            
#             # ATENÇÃO: Usando a nova função 'legacy' aqui
#             'Width': get_quantity_value_legacy(product, 'Width'),
#             'Thickness': get_quantity_value_legacy(product, 'Thickness'),
#             'Length': get_quantity_value_legacy(product, 'Length'),
#             'Height': get_quantity_value_legacy(product, 'Height'),
#         }
        
#         element_data.append(element_info)

#     df_ifc_data = pd.DataFrame(element_data)

#     desired_order = [
#         'Class', 'PredefinedType', 'BuildingStorey', 'Material', 'Name',
#         'PSET_RÔGGA.RÔGGA_SEÇÃO', 'PSET_RÔGGA.RÔGGA_DESCRIÇÃO',
#         'Width', 'Thickness', 'Length', 'Height',
#         'FileName', 'GlobalId'
#     ]

#     for col in desired_order:
#         if col not in df_ifc_data.columns:
#             df_ifc_data[col] = None
            
#     df_ifc_data = df_ifc_data[desired_order]

#     print("\nDataset criado com sucesso! Amostra dos dados:")
#     display(df_ifc_data.head().fillna(''))

# except FileNotFoundError:
#     print(f"ERRO: O arquivo não foi encontrado em: {ifc_file_path}")
# except Exception as e:
#     print(f"Ocorreu um erro inesperado: {e}")

In [4]:
# import ifcopenshell
# import ifcopenshell.util.element
# import pandas as pd
# import os

# # Tenta importar o 'display' para notebooks, se não, usará 'print'
# try:
#     from IPython.display import display
# except ImportError:
#     display = print # Faz 'display' funcionar como 'print' em ambientes não-notebook

# # ==============================================================================
# # --- CAMINHOS DE ENTRADA E SAÍDA ---
# # ==============================================================================

# # 1. Altere aqui para a PASTA que contém seus arquivos .ifc
# ifc_folder_path = r"C:\Users\lucas.galicioli\Downloads\RÔGGA EMPREENDIMENTOS-BRUSQUE HOME CLUB-2025-10-16-13-35-31-469"

# # 2. Defina onde você quer salvar o dataset consolidado
# output_csv_path = os.path.join(ifc_folder_path, "dataset_ifc_consolidado.csv")

# # ==============================================================================
# # --- Funções Auxiliares (Exatamente como as suas, não precisam de mudança) ---
# # ==============================================================================

# def get_building_storey(element):
#     """ Encontra o IfcBuildingStorey no qual o elemento está contido. """
#     try:
#         spatial_container = ifcopenshell.util.element.get_container(element)
#         if spatial_container and spatial_container.is_a('IfcBuildingStorey'):
#             return spatial_container.Name
#     except Exception:
#         pass
#     return None

# def get_material_name(element):
#     """ Extrai o nome do material associado ao elemento. """
#     material = ifcopenshell.util.element.get_material(element)
#     if not material:
#         return None
#     if hasattr(material, 'Name'):
#         return material.Name
#     elif hasattr(material, 'MaterialLayers'):
#         layer_names = [
#             layer.Material.Name 
#             for layer in material.MaterialLayers 
#             if hasattr(layer, 'Material') and hasattr(layer.Material, 'Name')
#         ]
#         return ', '.join(layer_names) if layer_names else None
#     return None
    
# def get_quantity_value_legacy(element, quantity_name):
#     """
#     Busca por uma quantidade específica (ex: 'Width') e retorna seu valor.
#     Esta versão é compatível com versões mais antigas do ifcopenshell sem 'get_qsets'.
#     """
#     # Itera através das relações de definição do elemento
#     for definition in getattr(element, 'IsDefinedBy', []):
#         if definition.is_a('IfcRelDefinesByProperties'):
#             prop_set = definition.RelatingPropertyDefinition
#             # Verifica se é um conjunto de quantidades (IfcElementQuantity)
#             if prop_set.is_a('IfcElementQuantity'):
#                 # Itera através das quantidades dentro do conjunto
#                 for quantity in prop_set.Quantities:
#                     if quantity.Name == quantity_name:
#                         # Extrai o valor do atributo correto (ex: LengthValue, AreaValue)
#                         value_attribute = next((attr for attr in dir(quantity) if attr.endswith('Value')), None)
#                         if value_attribute:
#                             return getattr(quantity, value_attribute)
#     return None

# # ==============================================================================
# # --- Processamento Principal (Modificado para Múltiplos Arquivos) ---
# # ==============================================================================

# # Lista principal para armazenar dados de TODOS os arquivos
# all_elements_data = []
# processed_files_count = 0

# print(f"Iniciando varredura da pasta: {ifc_folder_path}\n")

# # --- 1. Loop principal por todos os arquivos na pasta ---
# if not os.path.isdir(ifc_folder_path):
#     print(f"ERRO: O caminho especificado não é uma pasta válida: {ifc_folder_path}")
# else:
#     for filename in os.listdir(ifc_folder_path):
        
#         # --- 2. Verifica se é um arquivo IFC ---
#         if filename.lower().endswith('.ifc'):
#             ifc_file_path = os.path.join(ifc_folder_path, filename)
            
#             # --- 3. Bloco Try/Except para CADA arquivo ---
#             # Isso garante que um arquivo corrompido não pare todo o script
#             try:
#                 ifc_file = ifcopenshell.open(ifc_file_path)
#                 products = ifc_file.by_type('IfcProduct')
#                 print(f"Processando arquivo: {filename} ({len(products)} produtos encontrados)")

#                 # --- 4. Loop interno (como o original) para cada produto no arquivo ---
#                 for product in products:
#                     if product.is_a('IfcOpeningElement') or product.is_a('IfcVirtualElement'):
#                         continue

#                     psets = ifcopenshell.util.element.get_psets(product)
#                     rogga_pset = psets.get('PSET_RÔGGA', {})

#                     element_info = {
#                         'GlobalId': product.GlobalId,
#                         'FileName': filename, # Adiciona o nome do arquivo de origem
#                         'Class': product.is_a(),
#                         'PredefinedType': getattr(product, 'PredefinedType', None),
#                         'Name': getattr(product, 'Name', None),
#                         'BuildingStorey': get_building_storey(product),
#                         'Material': get_material_name(product),
#                         'PSET_RÔGGA.RÔGGA_SEÇÃO': rogga_pset.get('RÔGGA_SEÇÃO', None),
#                         'PSET_RÔGGA.RÔGGA_DESCRIÇÃO': rogga_pset.get('RÔGGA_DESCRIÇÃO', None),
                        
#                         # Usando a função 'legacy'
#                         'Width': get_quantity_value_legacy(product, 'Width'),
#                         'Thickness': get_quantity_value_legacy(product, 'Thickness'),
#                         'Length': get_quantity_value_legacy(product, 'Length'),
#                         'Height': get_quantity_value_legacy(product, 'Height'),
#                     }
                    
#                     # Adiciona os dados do elemento à lista principal
#                     all_elements_data.append(element_info)
                
#                 processed_files_count += 1

#             except Exception as e:
#                 print(f"   [ERRO] Falha ao processar o arquivo {filename}. Erro: {e}")
#                 print("   O arquivo pode estar corrompido ou não ser um IFC válido. Pulando...")

# # --- 5. Criação do DataFrame (APÓS processar todos os arquivos) ---
# if not all_elements_data:
#     print("\nProcessamento concluído, mas nenhum dado de elemento foi extraído.")
#     print("Verifique se a pasta contém arquivos .ifc válidos.")
# else:
#     print(f"\nProcessamento de {processed_files_count} arquivo(s) concluído.")
#     print(f"Total de {len(all_elements_data)} elementos extraídos.")
    
#     # Cria o DataFrame consolidado
#     df_ifc_data = pd.DataFrame(all_elements_data)

#     # --- 6. Ordenação e limpeza das colunas (como o original) ---
#     desired_order = [
#         'Class', 'PredefinedType', 'BuildingStorey', 'Material', 'Name',
#         'PSET_RÔGGA.RÔGGA_SEÇÃO', 'PSET_RÔGGA.RÔGGA_DESCRIÇÃO',
#         'Width', 'Thickness', 'Length', 'Height',
#         'FileName', 'GlobalId'
#     ]

#     # Garante que todas as colunas desejadas existam (preenche com None se faltar)
#     for col in desired_order:
#         if col not in df_ifc_data.columns:
#             df_ifc_data[col] = None
            
#     # Reordena o DataFrame
#     df_ifc_data = df_ifc_data[desired_order]

#     # --- 7. Exibir amostra e Salvar ---
#     print("\nDataset consolidado criado com sucesso! Amostra dos dados:")
#     display(df_ifc_data.head().fillna(''))

#     # --- 8. Salvar em CSV ---
#     try:
#         # 'encoding='utf-8-sig'' é bom para compatibilidade com Excel
#         df_ifc_data.to_csv(output_csv_path, index=False, encoding='utf-8-sig')
#         print(f"\nDataset salvo com sucesso em: {output_csv_path}")
#     except Exception as e:
#         print(f"\n[ERRO] Falha ao salvar o arquivo CSV em {output_csv_path}.")
#         print(f"   Erro: {e}")
#         print("   Verifique se você tem permissão de escrita no local.")

In [5]:
# df_matrix_item = pd.read_excel(r'C:\Users\lucas.galicioli\ifc-classifier\data\interim\classification-matrix-item.xlsx')

# df_ifc_data = pd.read_csv(r"C:\Users\lucas.galicioli\Downloads\RÔGGA EMPREENDIMENTOS-BRUSQUE HOME CLUB-2025-10-16-13-35-31-469\dataset_ifc_consolidado.csv")

# display(df_matrix_item)

In [6]:
# df_matrix = pd.merge(df_ifc_data, df_matrix_item, left_on="GlobalId", right_on="GUID", how="left")

In [7]:
# display(df_matrix)

In [8]:
# import pandas as pd
# import numpy as np
# import os

# # --- Nome da sua coluna ---
# # (Copiado da sua imagem, cuidado com o caractere 'Ô')
# coluna_alvo = "Ô_CLS_CLASSIFICAÇÃO_SOLIBRI" 

# # --- Caminho de destino ---
# pasta_destino = r"C:\Users\lucas.galicioli\ifc-classifier\data\interim"
# nome_arquivo_saida = "cls_matrix_v1.csv" # Nome do arquivo de saída
# caminho_completo_saida = os.path.join(pasta_destino, nome_arquivo_saida)

# # ==============================================================================
# # 1. FILTRAR O DATAFRAME
# # ==============================================================================
# # O Pandas 'notna()' seleciona apenas as linhas onde a coluna_alvo NÃO é nula.
# df_preenchidos = df_matrix[df_matrix[coluna_alvo].notna()]

# # ==============================================================================
# # 2. SALVAR O NOVO DATAFRAME EM CSV
# # ==============================================================================
# try:
#     # Garante que a pasta de destino exista
#     os.makedirs(pasta_destino, exist_ok=True)
    
#     # Salva o DataFrame filtrado (agora com os preenchidos)
#     # 'encoding='utf-8-sig'' é recomendado para boa compatibilidade com Excel
#     df_preenchidos.to_csv(caminho_completo_saida, index=False, encoding='utf-8-sig')
    
#     print(f"Arquivo com {len(df_preenchidos)} linhas PREENCHIDAS salvo com sucesso em:")
#     print(caminho_completo_saida)

# except KeyError:
#     print(f"ERRO: A coluna '{coluna_alvo}' não foi encontrada no DataFrame.")
# except Exception as e:
#     print(f"Ocorreu um erro ao tentar salvar o arquivo: {e}")

# # ==============================================================================
# # 3. (OPCIONAL) MOSTRAR UMA AMOSTRA
# # ==============================================================================
# print("\nAmostra dos dados filtrados (com classificação preenchida):")
# # Tenta usar display() se estiver em um notebook (Jupyter, Colab)
# try:
#     from IPython.display import display
#     display(df_preenchidos.head())
# except ImportError:
#     print(df_preenchidos.head().to_string())

In [9]:
df_matrix = pd.read_csv(r"C:\Users\lucas.galicioli\ifc-classifier\data\interim\cls_matrix_v1.csv")
df_matrix.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 311763 entries, 0 to 311762
Data columns (total 15 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Class                        311763 non-null  object 
 1   PredefinedType               232368 non-null  object 
 2   BuildingStorey               305822 non-null  object 
 3   Material                     168265 non-null  object 
 4   Name                         311247 non-null  object 
 5   PSET_RÔGGA.RÔGGA_SEÇÃO       135208 non-null  object 
 6   PSET_RÔGGA.RÔGGA_DESCRIÇÃO   203962 non-null  object 
 7   Width                        29264 non-null   float64
 8   Thickness                    0 non-null       float64
 9   Length                       109955 non-null  float64
 10  Height                       27144 non-null   float64
 11  FileName                     311763 non-null  object 
 12  GlobalId                     311763 non-null  object 
 13 

In [10]:
df_matrix.describe()

,Width,Thickness,Length,Height
count,29264.000000,0.0,109955.000000,27144.000000
mean,3.796640,NaN,104.987219,29.103547
std,25.206752,NaN,327.553395,50.815407
min,0.020000,NaN,0.005027,0.080000
25%,0.080000,NaN,6.196000,10.000000
50%,0.200000,NaN,31.902008,10.000000
75%,0.200000,NaN,115.103288,20.000000
max,1012.638805,NaN,10244.091941,374.580000


In [11]:
df_matrix.isnull().sum()

Class                               0
PredefinedType                  79395
BuildingStorey                   5941
Material                       143498
Name                              516
PSET_RÔGGA.RÔGGA_SEÇÃO         176555
PSET_RÔGGA.RÔGGA_DESCRIÇÃO     107801
Width                          282499
Thickness                      311763
Length                         201808
Height                         284619
FileName                            0
GlobalId                            0
GUID                                0
Ô_CLS_CLASSIFICAÇÃO_SOLIBRI         0
dtype: int64

In [12]:
import numpy as np # Importante ter o numpy

# Lista das colunas de texto que queremos limpar
colunas_para_limpar = [
    'Material',
    'PSET_RÔGGA.RÔGGA_SEÇÃO',
    'PSET_RÔGGA.RÔGGA_DESCRIÇÃO'

]

# Valor que representa "ausente" no seu arquivo
valor_ausente = 'NaN'

# Novo valor que vamos colocar no lugar
novo_valor = 'Desconhecido'

# Este loop vai passar por cada coluna da lista e fazer a substituição
for coluna in colunas_para_limpar:
    print(f"Limpando a coluna: {coluna}...")
    df_matrix[coluna] = df_matrix[coluna].replace(valor_ausente, novo_valor)

print("\nLimpeza concluída!")

Limpando a coluna: Material...
Limpando a coluna: PSET_RÔGGA.RÔGGA_SEÇÃO...
Limpando a coluna: PSET_RÔGGA.RÔGGA_DESCRIÇÃO...

Limpeza concluída!


In [13]:
display(df_matrix['Material'].value_counts())

Material
Concreto C40                                                     19608
AÇO DIN 2440                                                     16738
Linha Corrugado PVC Reforçado Laranja Antichamas                 13234
<Unnamed>                                                        10837
PVC Marrom                                                        9234
                                                                 ...  
Piso emborrachado azul (grânulo de pneu pigmentado) - Aubicon        1
Piso:ROGGA_PISO_COMUM_ARGAMASSA+CERÂMICA-5cm                         1
wire_196088225                                                       1
wire_228153184                                                       1
 FIBRA ÓTICA                                                         1
Name: count, Length: 346, dtype: int64

In [14]:
# Seleciona as colunas de entrada (features) para o modelo
features_selecionadas = [
    'Class',
    'PredefinedType',
    'BuildingStorey',
    'Material',
    'PSET_RÔGGA.RÔGGA_SEÇÃO',
    'PSET_RÔGGA.RÔGGA_DESCRIÇÃO',
    'Width',
    'Thickness',
    'Length',
    'Height',
    'Ô_CLS_CLASSIFICAÇÃO_SOLIBRI'
]

X = df_matrix[features_selecionadas]

print("DataFrame 'X' criado com as features selecionadas.")

DataFrame 'X' criado com as features selecionadas.


In [15]:
# Aplica o One-Hot Encoding em todas as colunas de texto dentro de X
X_encoded = pd.get_dummies(X)

print("One-Hot Encoding concluído!")

One-Hot Encoding concluído!


In [16]:
# 1. Mostra as dimensões (linhas, colunas) do novo DataFrame.
# O número de colunas vai aumentar BASTANTE!
print(X_encoded.shape)

# 2. Mostra as 5 primeiras linhas do novo DataFrame para vermos a estrutura
print(X_encoded.head())

(311763, 2298)
   Width  Thickness  Length  Height  Class_IfcAirTerminal  Class_IfcAlarm  \
0    NaN        NaN     NaN     NaN                 False           False   
1    NaN        NaN     NaN     NaN                 False           False   
2    NaN        NaN     NaN     NaN                 False           False   
3    NaN        NaN     NaN     NaN                 False           False   
4    NaN        NaN     NaN     NaN                 False           False   

   Class_IfcAudioVisualAppliance  Class_IfcBeam  \
0                          False          False   
1                          False          False   
2                          False          False   
3                          False          False   
4                          False          False   

   Class_IfcBuildingElementProxy  Class_IfcCableCarrierFitting  ...  \
0                          False                         False  ...   
1                          False                         False  ...   
2 

In [17]:
# Lista das colunas numéricas
colunas_numericas = ['Width', 'Thickness', 'Length', 'Height']

# colunas_numericas = ['Width', 'Length']

# Este loop passa por cada coluna da lista
for coluna in colunas_numericas:
    # 1. Calcula a mediana da coluna
    mediana = X_encoded[coluna].median()
    
    # 2. Usa .fillna() para preencher os NaN com o valor da mediana
    X_encoded[coluna] = X_encoded[coluna].fillna(mediana)
    
    print(f"Valores nulos na coluna '{coluna}' preenchidos com a mediana ({mediana}).")

print("\nPreenchimento de dados numéricos concluído!")

Valores nulos na coluna 'Width' preenchidos com a mediana (0.2).
Valores nulos na coluna 'Thickness' preenchidos com a mediana (nan).
Valores nulos na coluna 'Length' preenchidos com a mediana (31.90200821230576).
Valores nulos na coluna 'Height' preenchidos com a mediana (10.000000000001192).

Preenchimento de dados numéricos concluído!


c:\Users\lucas.galicioli\ifc-classifier\.venv\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


In [18]:
# Soma todos os valores nulos em todas as colunas do DataFrame
total_nulos = X_encoded.isnull().sum().sum()

print(f"\nTotal de valores nulos no DataFrame final 'X_encoded': {total_nulos}")

if total_nulos == 0:
    print("Parabéns! Seus dados estão 100% limpos e prontos para o treinamento do modelo!")
else:
    print("Ainda existem valores nulos. Precisamos investigar o que aconteceu.")


Total de valores nulos no DataFrame final 'X_encoded': 311763
Ainda existem valores nulos. Precisamos investigar o que aconteceu.


In [19]:
from sklearn.preprocessing import LabelEncoder

# 1. Seleciona a primeira coluna que queremos prever do DataFrame original
y1 = df_matrix['Ô_CLS_CLASSIFICAÇÃO_SOLIBRI']

# 2. Cria uma instância do codificador
le1 = LabelEncoder()

# 3.Ajusta o codificador aos seus dados e os transforma em números
y1_encoded = le1.fit_transform(y1)

print("Variável alvo 'Ô_CLS_CLASSIFICAÇÃO_SOLIBRI' foi codificada com sucesso!")
print("Exemplo dos primeiros 5 valores codificados:", y1_encoded[:5])


Variável alvo 'Ô_CLS_CLASSIFICAÇÃO_SOLIBRI' foi codificada com sucesso!
Exemplo dos primeiros 5 valores codificados: [137 137 137  56  56]


In [20]:
# Certifique-se de que o numpy está importado (geralmente como np)
import numpy as np

print("\nPasso 2.5 de 3: Removendo classes com apenas 1 amostra...")

# 1. Encontre as classes únicas e suas contagens (o equivalente do NumPy para value_counts)
classes, class_counts = np.unique(y1_encoded, return_counts=True)

# 2. Identifique as classes que têm 2 ou mais amostras
#    (Filtra o array de 'classes' usando o array de 'class_counts')
valid_classes = classes[class_counts >= 2]

# 3. Crie uma "máscara" (usando np.isin) para filtrar o array original
mask = np.isin(y1_encoded, valid_classes)

# 4. Filtre X_encoded e y1_encoded usando a máscara
#    (Funciona tanto para DataFrames (X) quanto para arrays (y))
X_encoded_filtered = X_encoded[mask]
y1_encoded_filtered = y1_encoded[mask]

print(f"--> Amostras removidas: {len(y1_encoded) - len(y1_encoded_filtered)}")
print(f"--> Shape de X antes da filtragem: {X_encoded.shape}")
print(f"--> Shape de X após a filtragem: {X_encoded_filtered.shape}")
print(f"--> Shape de y antes da filtragem: {y1_encoded.shape}")
print(f"--> Shape de y após a filtragem: {y1_encoded_filtered.shape}")

# ------------------------------------------------------------------------------

# AGORA, no Passo 3, use os dataframes filtrados:
print("\nPasso 3 de 3: Dividindo os dados em conjuntos de treino e teste...")

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded_filtered,   # Use a versão filtrada
    y1_encoded_filtered,  # Use a versão filtrada
    test_size=0.25,
    random_state=42,
    stratify=y1_encoded_filtered # Estratifique pelo y filtrado
)

print("--> Dados divididos com sucesso!")
print(f"--> Shape do X_train final: {X_train.shape}")
print(f"--> Shape do y_train final: {y_train.shape}")
print("\n✅ PREPARAÇÃO CONCLUÍDA.")


Passo 2.5 de 3: Removendo classes com apenas 1 amostra...
--> Amostras removidas: 2
--> Shape de X antes da filtragem: (311763, 2298)
--> Shape de X após a filtragem: (311761, 2298)
--> Shape de y antes da filtragem: (311763,)
--> Shape de y após a filtragem: (311761,)

Passo 3 de 3: Dividindo os dados em conjuntos de treino e teste...
--> Dados divididos com sucesso!
--> Shape do X_train final: (233820, 2298)
--> Shape do y_train final: (233820,)

✅ PREPARAÇÃO CONCLUÍDA.


In [21]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder # <-- Importe o LabelEncoder aqui

# ==============================================================================
# BLOCO DE CÓDIGO CONSOLIDADO (COM A CORREÇÃO)
# ==============================================================================

print("Passo 1 de 4: Aplicando One-Hot Encoding em X...")
# X e y1_encoded vêm da Célula 13
X_encoded = pd.get_dummies(X)
X_encoded.columns = X_encoded.columns.str.replace(r'\\[|\\]|<', '_', regex=True)
print(f"--> DataFrame 'X_encoded' criado com shape: {X_encoded.shape}")

# ------------------------------------------------------------------------------

print("\nPasso 2 de 4: Removendo classes com 1 amostra...")
# y1_encoded e le1 (o encoder "master") vêm da Célula 13

classes, class_counts = np.unique(y1_encoded, return_counts=True)
valid_classes = classes[class_counts >= 2]
mask = np.isin(y1_encoded, valid_classes)

X_encoded_filtered = X_encoded[mask]
y1_encoded_filtered = y1_encoded[mask] # <-- Este 'y' tem "buracos" (ex: 0, 1, 3, 4)

print(f"--> Amostras removidas: {len(y1_encoded) - len(y1_encoded_filtered)}")
print(f"--> Shape de X após a filtragem: {X_encoded_filtered.shape}")

# ------------------------------------------------------------------------------

print("\nPasso 3 de 4: Re-codificando 'y' para XGBoost (Removendo Gaps)...")
# ESTA É A CORREÇÃO CRUCIAL
# Nós criamos um NOVO encoder (le_xgb) que transforma os rótulos com "buracos"
# (ex: [0, 1, 3, 4]) em um rótulo contínuo que o XGBoost aceita (ex: [0, 1, 2, 3]).

le_xgb = LabelEncoder()
y_xgb_encoded = le_xgb.fit_transform(y1_encoded_filtered)

print(f"--> 'y' re-codificado. Novo range de classes: 0 a {y_xgb_encoded.max()}")
print(f"--> Total de classes únicas para o modelo: {len(le_xgb.classes_)}")

# ------------------------------------------------------------------------------

print("\nPasso 4 de 4: Dividindo os dados em conjuntos de treino e teste...")
# AGORA, usamos o X filtrado e o NOVO y_xgb_encoded

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded_filtered,   # <-- Use o X filtrado
    y_xgb_encoded,        # <-- Use o NOVO 'y' re-codificado
    test_size=0.25,
    random_state=42,
    stratify=y_xgb_encoded  # <-- Estratifique pelo NOVO 'y'
)

print("--> Dados divididos com sucesso!")
print(f"--> Shape do X_train final: {X_train.shape}")
print(f"--> Shape do y_train final: {y_train.shape}")
print("\n✅ PREPARAÇÃO CONCLUÍDA. Agora você pode rodar a célula de treinamento.")
# ==============================================================================

Passo 1 de 4: Aplicando One-Hot Encoding em X...
--> DataFrame 'X_encoded' criado com shape: (311763, 2298)

Passo 2 de 4: Removendo classes com 1 amostra...
--> Amostras removidas: 2
--> Shape de X após a filtragem: (311761, 2298)

Passo 3 de 4: Re-codificando 'y' para XGBoost (Removendo Gaps)...
--> 'y' re-codificado. Novo range de classes: 0 a 139
--> Total de classes únicas para o modelo: 140

Passo 4 de 4: Dividindo os dados em conjuntos de treino e teste...
--> Dados divididos com sucesso!
--> Shape do X_train final: (233820, 2298)
--> Shape do y_train final: (233820,)

✅ PREPARAÇÃO CONCLUÍDA. Agora você pode rodar a célula de treinamento.


In [22]:
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report
import numpy as np # <-- CERTIFIQUE-SE DE QUE NUMPY ESTÁ IMPORTADO

# ==============================================================================
# BLOCO DE CÓDIGO PARA TREINAMENTO E AVALIAÇÃO RÁPIDA (COM CORREÇÃO FINAL)
# ==============================================================================

# 1. Cria a instância do classificador XGBoost
# Esta parte não muda.
xgb_classifier = xgb.XGBClassifier(
    objective='multi:softmax',
    tree_method="hist", 
    device="cuda",
    use_label_encoder=False,
    eval_metric='mlogloss'
)

# 2. Treina o modelo com os dados de treino
print("Iniciando o treinamento do modelo XGBoost (versão rápida)...")
# Esta chamada agora usa o X_train e y_train (contínuo) da célula anterior
xgb_classifier.fit(X_train, y_train)
print("Treinamento concluído!")

# ------------------------------------------------------------------------------

# 3. Usa o modelo treinado para fazer previsões nos dados de teste
print("\nRealizando previsões nos dados de teste...")
y_pred = xgb_classifier.predict(X_test)

# 4. Avalia a acurácia do modelo
accuracy = accuracy_score(y_test, y_pred)
print(f"\nAcurácia do modelo: {accuracy:.4f}")
print(f"Isso significa que o modelo acertou {accuracy:.2%} das classificações!")

# 5. Gera um relatório de classificação detalhado
# --- INÍCIO DA CORREÇÃO PARA O VALUEERROR ---

# 1. Pega as classes do encoder do XGB (que são os rótulos originais com "gaps")
#    (le_xgb foi criado na célula 15)
original_gappy_labels = le_xgb.classes_

# 2. Use o encoder "master" (le1) para transformar esses números de volta em nomes
#    (le1 foi criado na célula 13)
target_names_corretos = le1.inverse_transform(original_gappy_labels)

# --- INÍCIO DA CORREÇÃO DO TYPEERROR ---
# A função classification_report espera que 'target_names' seja uma lista de
# strings. O erro 'TypeError: object of type 'float' has no len()'
# indica que le1.inverse_transform() está retornando números.
# Convertemos explicitamente para string aqui:
target_names_corretos = target_names_corretos.astype(str)
# --- FIM DA CORREÇÃO DO TYPEERROR ---


# 3. CRIE A LISTA DE LABELS (OS NÚMEROS CONTÍNUOS DE 0 a N-1)
#    Esta é a correção principal. Ela força o relatório a usar todas as 125 classes.
labels_para_report = np.arange(len(target_names_corretos))


print("\nRelatório de Classificação Detalhado:")
# 4. PASSE OS LABELS E OS NOMES. Agora eles têm o mesmo tamanho (125).
print(classification_report(
    y_test, 
    y_pred, 
    labels=labels_para_report,          # <-- PARÂMETRO ADICIONADO
    target_names=target_names_corretos, 
    zero_division=0
))
# ==============================================================================

Iniciando o treinamento do modelo XGBoost (versão rápida)...


c:\Users\lucas.galicioli\ifc-classifier\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [14:53:29] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Treinamento concluído!

Realizando previsões nos dados de teste...


c:\Users\lucas.galicioli\ifc-classifier\.venv\Lib\site-packages\xgboost\core.py:729: UserWarning: [15:10:26] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)



Acurácia do modelo: 1.0000
Isso significa que o modelo acertou 100.00% das classificações!

Relatório de Classificação Detalhado:
                                                                                                    precision    recall  f1-score   support

                                                                                    Aba de Fachada       1.00      1.00      1.00        35
                                                                                Abrigo de hidrante       1.00      1.00      1.00        32
                                                                               Abrigo para medidor       1.00      1.00      1.00         2
                                                                                  Acionador manual       1.00      1.00      1.00        19
                                                                                       Acionadores       1.00      1.00      1.00        32
                            

In [23]:
# import xgboost as xgb
# from sklearn.metrics import accuracy_score, classification_report
# import numpy as np
# import matplotlib.pyplot as plt

# # ==============================================================================
# # BLOCO DE CÓDIGO COM A INTERFACE NATIVA (DMatrix) - MAIS EFICIENTE
# # ==============================================================================

# # 1. Converta seus dados para DMatrix
# #    Este objeto é otimizado para o XGBoost
# print("Convertendo dados para DMatrix...")
# dtrain = xgb.DMatrix(X_train, label=y_train)
# dtest = xgb.DMatrix(X_test, label=y_test)
# print("Conversão concluída.")

# # 2. Defina os parâmetros do modelo
# #    (O que estava no __init__ do XGBClassifier vai para um dict)
# #    PRECISAMOS ADICIONAR 'num_class'
# num_classes = len(np.unique(y_train)) # Isso deve ser 125 no seu caso
# print(f"Detectado {num_classes} classes.")

# params = {
#     'objective': 'multi:softmax',
#     'num_class': num_classes,      # <-- CRUCIAL para multi:softmax
#     'tree_method': "hist", 
#     'device': "cpu",            # <-- Tente com 'cuda' primeiro, se falhar, mude para 'cpu'
#     'eval_metric': 'mlogloss'
# }

# # 3. Defina os conjuntos de avaliação
# evals = [(dtrain, 'train'), (dtest, 'test')]

# print("Iniciando o treinamento do modelo XGBoost (nativo)...")

# # 4. Treine o modelo usando xgb.train
# #    'evals_result' armazena o histórico para o plot
# evals_result = {} # Dicionário para armazenar os logs
# bst = xgb.train(
#     params,
#     dtrain,
#     num_boost_round=500,          # Equivalente a n_estimators
#     evals=evals,
#     early_stopping_rounds=20,
#     evals_result=evals_result,    # Passa o dicionário
#     verbose_eval=50
# )

# print("Treinamento concluído!")

# # ------------------------------------------------------------------------------
# # 5. GERAÇÃO DO RECURSO VISUAL (CURVAS DE APRENDIZADO)
# #    (O código de plot muda um pouco para se adaptar ao dict 'evals_result')
# # ------------------------------------------------------------------------------
# print("\nGerando gráfico das Curvas de Aprendizado...")

# epochs = len(evals_result['train']['mlogloss'])
# x_axis = range(0, epochs)

# plt.figure(figsize=(10, 6))
# # As chaves agora são 'train' e 'test' (como definimos em 'evals')
# plt.plot(x_axis, evals_result['train']['mlogloss'], label='Erro de Treino')
# plt.plot(x_axis, evals_result['test']['mlogloss'], label='Erro de Validação (Teste)')
# plt.legend()
# plt.ylabel('Erro (mlogloss)')
# plt.xlabel('Rodada de Treinamento (Árvore)')
# plt.title('Curvas de Aprendizado do XGBoost (API Nativa)')
# plt.grid(True)
# plt.show()

# # ------------------------------------------------------------------------------

# # 6. Faça as previsões (usando o Booster 'bst')
# print("\nRealizando previsões nos dados de teste...")
# # O .predict da API nativa já retorna a classe (não probabilidades)
# # por causa do 'multi:softmax'
# y_pred = bst.predict(dtest)

# # 7. O resto do seu código de avaliação permanece O MESMO
# accuracy = accuracy_score(y_test, y_pred)
# print(f"\nAcurácia do modelo: {accuracy:.4f}")

# original_gappy_labels = le_xgb.classes_
# target_names_corretos = le1.inverse_transform(original_gappy_labels)
# target_names_corretos = target_names_corretos.astype(str)
# labels_para_report = np.arange(len(target_names_corretos))

# print("\nRelatório de Classificação Detalhado:")
# print(classification_report(
#     y_test, 
#     y_pred, 
#     labels=labels_para_report,
#     target_names=target_names_corretos, 
#     zero_division=0
# ))
# # ==============================================================================

In [24]:
# # ==============================================================================
# # NOVO BLOCO: EXIBINDO OS RESULTADOS DETALHADOS (REAL vs. PREVISTO)
# # ==============================================================================
# print("\n" + "="*80)
# print("VERIFICAÇÃO DETALHADA DAS PREVISÕES (REAL vs. PREVISTO)")
# print("="*80)

# try:
#     # 1. Importar Pandas para criar a tabela de visualização
#     import pandas as pd
#     pd.set_option('display.max_rows', 200) # Para mostrar mais linhas se necessário
#     pd.set_option('display.max_columns', 10) # Para mostrar mais colunas

#     # 2. Criar um mapa do ID numérico (0, 1, 2...) para o Nome da classe
#     #    labels_para_report = [0, 1, 2, ..., 124]
#     #    target_names_corretos = ["NomeA", "NomeB", "NomeC", ..., "NomeZ"]
#     mapa_id_para_nome = dict(zip(labels_para_report, target_names_corretos))

#     # 3. Criar o DataFrame principal de comparação
#     df_comparacao = pd.DataFrame({
#         'Real_ID': y_test,
#         'Previsto_ID': y_pred
#     })

#     # 4. Mapear os IDs de volta para os nomes originais para melhor leitura
#     df_comparacao['Real_Nome'] = df_comparacao['Real_ID'].map(mapa_id_para_nome)
#     df_comparacao['Previsto_Nome'] = df_comparacao['Previsto_ID'].map(mapa_id_para_nome)

#     # 5. Adicionar uma coluna para ver facilmente os acertos/erros
#     df_comparacao['Acertou?'] = df_comparacao['Real_ID'] == df_comparacao['Previsto_ID']

#     # 6. Mostrar as primeiras 25 previsões
#     print("\n[--- Amostra das Previsões (primeiras 25) ---]")
    
#     # Tenta usar display() se estiver em um notebook (Jupyter, Colab)
#     # Senão, usa print() normal, que formata bem o DataFrame.
#     try:
#         from IPython.display import display
#         display(df_comparacao.head(25))
#     except ImportError:
#         # .to_string() garante uma boa formatação no console padrão
#         print(df_comparacao.head(25).to_string()) 

#     # 7. Mostrar APENAS os erros (se houver)
#     #    (No seu caso, com 100% de acurácia, esta tabela estará vazia)
#     df_erros = df_comparacao[df_comparacao['Acertou?'] == False]
    
#     if df_erros.empty:
#         print("\n[--- Análise de Erros ---]")
#         print("Parabéns! O modelo não cometeu erros no conjunto de teste.")
#     else:
#         print(f"\n[--- Exibindo os {len(df_erros)} Erros de Classificação ---]")
#         try:
#             from IPython.display import display
#             display(df_erros)
#         except ImportError:
#             print(df_erros.to_string())

# except ImportError:
#     # Fallback caso o usuário não tenha pandas instalado
#     print("\nPara uma visualização detalhada em tabela, instale a biblioteca pandas:")
#     print("pip install pandas")
#     print("\nResultados brutos (sem pandas - 20 primeiras amostras):")
#     print(f"Valores Reais (y_test):    {y_test[:20]}...")
#     print(f"Valores Previstos (y_pred): {y_pred[:20]}...")
# # ==============================================================================

In [30]:
import joblib

# Salva o modelo treinado em um arquivo
joblib.dump(xgb_classifier, 'ifc_classifier_disciplinas_v1.pkl')

# Salva também o LabelEncoder, pois você precisará dele para decodificar as previsões
joblib.dump(le1, 'label_encoder_disciplinas_v1.pkl')

print("Modelo e LabelEncoder salvos com sucesso!")

Modelo e LabelEncoder salvos com sucesso!


In [31]:
import json

# --- SALVAR ARTEFATOS ADICIONAIS ---

# 3. Salvar a lista de colunas do modelo
colunas_do_modelo = X_encoded.columns.tolist()
with open('colunas_modelo_disciplinas.json', 'w') as f:
    json.dump(colunas_do_modelo, f)

# 4. Salvar as medianas de treinamento
colunas_numericas = ['Width', 'Thickness', 'Length', 'Height']
medianas = df_matrix[colunas_numericas].median().to_dict()
with open('medianas_treinamento.json', 'w') as f:
    json.dump(medianas, f)

print("Artefatos de pré-processamento (colunas e medianas) salvos com sucesso!")

Artefatos de pré-processamento (colunas e medianas) salvos com sucesso!


In [27]:
# import xgboost as xgb

# xgb_model = xgb.XGBClassifier(
#     objective='multi:softmax',
#     tree_method="hist", 
#     device="cuda",
#     use_label_encoder=False,
#     eval_metric='mlogloss'
# )

# grid_search = GridSearchCV(
#     estimator=xgb_model,
#     param_grid=param_grid,
#     cv=3,
#     scoring='accuracy',
#     n_jobs=-1,
#     verbose=2
# )

# print("Iniciando a busca pelos melhores hiperparâmetros com GridSearchCV...")
# grid_search.fit(X_train, y_train)

# print("\nBusca concluída!")


# # Mostra a melhor combinação de parâmetros encontrada
# print("Melhores parâmetros encontrados:")
# print(grid_search.best_params_)

# # Mostra a pontuação (acurácia) obtida com os melhores parâmetros durante a validação cruzada
# print("\nMelhor pontuação de acurácia (validação cruzada):")
# print(grid_search.best_score_)

In [28]:
# import ifcopenshell
# import ifcopenshell.util.element
# import pandas as pd
# import os

# # --- Caminho do arquivo IFC ---
# ifc_file_path = r"C:\Users\lucas.galicioli\Downloads\RÔGGA EMPREENDIMENTOS-BRUSQUE HOME CLUB-2025-10-16-13-35-31-469\PHN21043-PCI-PR-0001-BIM-EMB-GER SPK-R03.ifc"

# # --- Funções Auxiliares (com a função de quantidade modificada) ---

# def get_building_storey(element):
#     """ Encontra o IfcBuildingStorey no qual o elemento está contido. """
#     try:
#         spatial_container = ifcopenshell.util.element.get_container(element)
#         if spatial_container and spatial_container.is_a('IfcBuildingStorey'):
#             return spatial_container.Name
#     except Exception:
#         pass
#     return None

# def get_material_name(element):
#     """ Extrai o nome do material associado ao elemento. """
#     material = ifcopenshell.util.element.get_material(element)
#     if not material:
#         return None
#     if hasattr(material, 'Name'):
#         return material.Name
#     elif hasattr(material, 'MaterialLayers'):
#         layer_names = [
#             layer.Material.Name 
#             for layer in material.MaterialLayers 
#             if hasattr(layer, 'Material') and hasattr(layer.Material, 'Name')
#         ]
#         return ', '.join(layer_names) if layer_names else None
#     return None
    
# # --- SOLUÇÃO 2: Função de quantidade compatível com versões antigas ---
# def get_quantity_value_legacy(element, quantity_name):
#     """
#     Busca por uma quantidade específica (ex: 'Width') e retorna seu valor.
#     Esta versão é compatível com versões mais antigas do ifcopenshell sem 'get_qsets'.
#     """
#     # Itera através das relações de definição do elemento
#     for definition in getattr(element, 'IsDefinedBy', []):
#         if definition.is_a('IfcRelDefinesByProperties'):
#             prop_set = definition.RelatingPropertyDefinition
#             # Verifica se é um conjunto de quantidades (IfcElementQuantity)
#             if prop_set.is_a('IfcElementQuantity'):
#                 # Itera através das quantidades dentro do conjunto
#                 for quantity in prop_set.Quantities:
#                     if quantity.Name == quantity_name:
#                         # Extrai o valor do atributo correto (ex: LengthValue, AreaValue)
#                         value_attribute = next((attr for attr in dir(quantity) if attr.endswith('Value')), None)
#                         if value_attribute:
#                             return getattr(quantity, value_attribute)
#     return None

# # --- Processamento Principal ---

# try:
#     ifc_file = ifcopenshell.open(ifc_file_path)
#     file_name = os.path.basename(ifc_file_path)

#     element_data = []
#     products = ifc_file.by_type('IfcProduct')

#     print(f"Processando {len(products)} elementos do arquivo: {file_name}...")

#     for product in products:
#         if product.is_a('IfcOpeningElement') or product.is_a('IfcVirtualElement'):
#             continue

#         psets = ifcopenshell.util.element.get_psets(product)
#         rogga_pset = psets.get('PSET_RÔGGA', {})

#         element_info = {
#             'GlobalId': product.GlobalId,
#             'FileName': file_name,
#             'Class': product.is_a(),
#             'PredefinedType': getattr(product, 'PredefinedType', None),
#             'Name': getattr(product, 'Name', None),
#             'BuildingStorey': get_building_storey(product),
#             'Material': get_material_name(product),
#             'PSET_RÔGGA.RÔGGA_SEÇÃO': rogga_pset.get('RÔGGA_SEÇÃO', None),
#             'PSET_RÔGGA.RÔGGA_DESCRIÇÃO': rogga_pset.get('RÔGGA_DESCRIÇÃO', None),
            
#             # ATENÇÃO: Usando a nova função 'legacy' aqui
#             'Width': get_quantity_value_legacy(product, 'Width'),
#             'Thickness': get_quantity_value_legacy(product, 'Thickness'),
#             'Length': get_quantity_value_legacy(product, 'Length'),
#             'Height': get_quantity_value_legacy(product, 'Height'),
#         }
        
#         element_data.append(element_info)

#     df_ifc_data = pd.DataFrame(element_data)

#     desired_order = [
#         'Class', 'PredefinedType', 'BuildingStorey', 'Material', 'Name',
#         'PSET_RÔGGA.RÔGGA_SEÇÃO', 'PSET_RÔGGA.RÔGGA_DESCRIÇÃO',
#         'Width', 'Thickness', 'Length', 'Height',
#         'FileName', 'GlobalId'
#     ]

#     for col in desired_order:
#         if col not in df_ifc_data.columns:
#             df_ifc_data[col] = None
            
#     df_ifc_data = df_ifc_data[desired_order]

#     print("\nDataset criado com sucesso! Amostra dos dados:")
#     display(df_ifc_data.head().fillna(''))

# except FileNotFoundError:
#     print(f"ERRO: O arquivo não foi encontrado em: {ifc_file_path}")
# except Exception as e:
#     print(f"Ocorreu um erro inesperado: {e}")

In [29]:
# import ifcopenshell
# import ifcopenshell.util.element
# import pandas as pd
# import os

# # --- Caminho do arquivo IFC ---
# ifc_file_path = r"C:\Users\lucas.galicioli\Downloads\RÔGGA EMPREENDIMENTOS-BRUSQUE HOME CLUB-2025-10-16-13-35-31-469\PHN21043-ARQ-EX-0002-BIM-TOR-GER-R01.ifc"

# # --- Caminho da sua matriz de classificação ---
# # --- NOVO ---
# path_to_matrix = r'C:\Users\lucas.galicioli\ifc-classifier\data\interim\classification-matrix.xlsx'
# coluna_matrix_filename = 'FileName'            # Nome da coluna de arquivos na matriz
# coluna_matrix_disciplina = 'Ô_CLS_DISCIPLINAS' # Nome da coluna de disciplina na matriz


# # --- Funções Auxiliares ---

# def get_building_storey(element):
#     """ Encontra o IfcBuildingStorey no qual o elemento está contido. """
#     try:
#         spatial_container = ifcopenshell.util.element.get_container(element)
#         if spatial_container and spatial_container.is_a('IfcBuildingStorey'):
#             return spatial_container.Name
#     except Exception:
#         pass
#     return None

# def get_material_name(element):
#     """ Extrai o nome do material associado ao elemento. """
#     material = ifcopenshell.util.element.get_material(element)
#     if not material:
#         return None
#     if hasattr(material, 'Name'):
#         return material.Name
#     elif hasattr(material, 'MaterialLayers'):
#         layer_names = [
#             layer.Material.Name 
#             for layer in material.MaterialLayers 
#             if hasattr(layer, 'Material') and hasattr(layer.Material, 'Name')
#         ]
#         return ', '.join(layer_names) if layer_names else None
#     return None
    
# def get_quantity_value_legacy(element, quantity_name):
#     """
#     Busca por uma quantidade específica (ex: 'Width') e retorna seu valor.
#     """
#     for definition in getattr(element, 'IsDefinedBy', []):
#         if definition.is_a('IfcRelDefinesByProperties'):
#             prop_set = definition.RelatingPropertyDefinition
#             if prop_set.is_a('IfcElementQuantity'):
#                 for quantity in prop_set.Quantities:
#                     if quantity.Name == quantity_name:
#                         value_attribute = next((attr for attr in dir(quantity) if attr.endswith('Value')), None)
#                         if value_attribute:
#                             return getattr(quantity, value_attribute)
#     return None

# # --- NOVAS FUNÇÕES AUXILIARES PARA MAPEAMENTO ---

# def gerar_mapa_disciplinas(matrix_path, col_filename, col_disciplina):
#     """
#     Carrega a matriz de classificação e cria um dicionário de mapeamento
#     (Código -> Disciplina). Ex: {'HID': 'Hidrossanitário', 'EST': 'Estrutura'}
#     """
#     try:
#         df_matrix = pd.read_excel(matrix_path)
        
#         # Criar df temporário apenas com as colunas necessárias
#         df_mapa_temp = df_matrix[[col_filename, col_disciplina]].copy()
        
#         # Extrair o código (ex: "EST", "HID")
#         df_mapa_temp['Disciplina_Code'] = df_mapa_temp[col_filename].str.split('-').str[1]
        
#         # Limpar (remover nulos e duplicatas)
#         df_mapa_temp = df_mapa_temp[['Disciplina_Code', col_disciplina]].dropna().drop_duplicates()
        
#         # Converter para dicionário
#         mapa = df_mapa_temp.set_index('Disciplina_Code')[col_disciplina].to_dict()
        
#         if not mapa:
#             print(f"AVISO: O mapa de disciplinas gerado a partir de '{matrix_path}' está vazio.")
        
#         return mapa
    
#     except FileNotFoundError:
#         print(f"ERRO: Arquivo da matriz não encontrado em: {matrix_path}")
#         return None
#     except KeyError as e:
#         print(f"ERRO: Coluna {e} não encontrada na matriz. Verifique os nomes '{col_filename}' e '{col_disciplina}'.")
#         return None
#     except Exception as e:
#         print(f"ERRO ao gerar mapa de disciplinas: {e}")
#         return None

# def extrair_disciplina_do_nome(nome_arquivo_base, mapa_disciplinas):
#     """
#     Extrai o código do nome do arquivo e o traduz usando o mapa.
#     """
#     try:
#         nome_base = os.path.splitext(nome_arquivo_base)[0]
#         partes_nome = nome_base.split('-')
#         codigo_disciplina = partes_nome[1] # Pega o "HID", "EST", etc.
        
#         # Traduz usando o mapa
#         disciplina_traduzida = mapa_disciplinas.get(codigo_disciplina, f"Código '{codigo_disciplina}' Não Mapeado")
#         return disciplina_traduzida
#     except IndexError:
#         print(f"AVISO: O nome '{nome_arquivo_base}' não segue o padrão 'XXX-CODIGO-...'")
#         return "Erro: Padrão de Nome"
#     except Exception:
#         return "Erro: Extração"

# # --- Processamento Principal ---

# try:
#     ifc_file = ifcopenshell.open(ifc_file_path)
#     file_name = os.path.basename(ifc_file_path)

#     element_data = []
#     products = ifc_file.by_type('IfcProduct')

#     print(f"Processando {len(products)} elementos do arquivo: {file_name}...")

#     for product in products:
#         if product.is_a('IfcOpeningElement') or product.is_a('IfcVirtualElement'):
#             continue

#         psets = ifcopenshell.util.element.get_psets(product)
#         rogga_pset = psets.get('PSET_RÔGGA', {})

#         element_info = {
#             'GlobalId': product.GlobalId,
#             'FileName': file_name,
#             'Class': product.is_a(),
#             'PredefinedType': getattr(product, 'PredefinedType', None),
#             'Name': getattr(product, 'Name', None),
#             'BuildingStorey': get_building_storey(product),
#             'Material': get_material_name(product),
#             'PSET_RÔGGA.RÔGGA_SEÇÃO': rogga_pset.get('RÔGGA_SEÇÃO', None),
#             'PSET_RÔGGA.RÔGGA_DESCRIÇÃO': rogga_pset.get('RÔGGA_DESCRIÇÃO', None),
#             'Width': get_quantity_value_legacy(product, 'Width'),
#             'Thickness': get_quantity_value_legacy(product, 'Thickness'),
#             'Length': get_quantity_value_legacy(product, 'Length'),
#             'Height': get_quantity_value_legacy(product, 'Height'),
#         }
        
#         element_data.append(element_info)

#     df_ifc_data = pd.DataFrame(element_data)

#     # --- INÍCIO DA LÓGICA DE CLASSIFICAÇÃO DE DISCIPLINA ---
#     # --- NOVO ---
#     print("\nDataFrame criado. Gerando mapa e adicionando classificação de Disciplina...")

#     # 1. Gerar o mapa de disciplinas (carregando a matriz)
#     mapa_disciplinas = gerar_mapa_disciplinas(path_to_matrix, coluna_matrix_filename, coluna_matrix_disciplina)

#     # 2. Extrair e traduzir a disciplina do arquivo atual
#     disciplina_final = "Erro ao Mapear" # Valor padrão
#     if mapa_disciplinas:
#         # Usa a variável 'file_name' que já definimos no começo do 'try'
#         disciplina_final = extrair_disciplina_do_nome(file_name, mapa_disciplinas)
#     else:
#         print("AVISO: Não foi possível carregar o mapa. A disciplina não será adicionada corretamente.")

#     # 3. Adicionar a coluna ao DataFrame
#     #    (O mesmo valor será aplicado a todas as linhas, o que está correto)
#     df_ifc_data['Ô_CLS_DISCIPLINAS'] = disciplina_final
#     print(f"Ô_CLS_DISCIPLINAS '{disciplina_final}' atribuída a {len(df_ifc_data)} elementos.")
    
#     # --- FIM DA LÓGICA DE CLASSIFICAÇÃO ---


#     # --- Reordenação das Colunas ---
#     desired_order = [
#         'Class',
#         'PredefinedType', 'BuildingStorey', 'Material', 'Name',
#         'PSET_RÔGGA.RÔGGA_SEÇÃO', 'PSET_RÔGGA.RÔGGA_DESCRIÇÃO',
#         'Width', 'Thickness', 'Length', 'Height',
#         'FileName',
#         'Ô_CLS_DISCIPLINAS', # Coluna nova adicionada à lista
#         'GlobalId'
#     ]

#     # Garante que todas as colunas existem (bom para robustez)
#     for col in desired_order:
#         if col not in df_ifc_data.columns:
#             df_ifc_data[col] = None
            
#     df_ifc_data = df_ifc_data[desired_order]

#     print("\nDataset criado com sucesso! Amostra dos dados (com Ô_CLS_DISCIPLINAS):")
#     display(df_ifc_data)

# except FileNotFoundError:
#     print(f"ERRO: O arquivo não foi encontrado em: {ifc_file_path}")
# except Exception as e:
#     print(f"Ocorreu um erro inesperado: {e}")